In [ ]:
# Install libraries
#import tabula
import os
import pandas as pd
import numpy as np
import re
import glob
from datetime import datetime
import sys
import csv
import openpyxl
import matplotlib.pyplot as plt


# Set Working Directory
os.chdir(r'C:/Users/luke.gordon/OneDrive - Resource Systems Group, Inc\Sharepoint - Model Development/CCRPC (VT) Model Update/Data/Household-Travel-Data/')

#!pip install pycryptodome



## Function Set

In [ ]:
# Transpose and clean table function
def transpose_tbl(in_tbl):
    out_tbl = in_tbl.set_index("Label (Grouping)").T   # Transposes the table and sets the index to the "Label (Grouping)" column
    out_tbl.columns = out_tbl.columns.str.strip().str.replace(r'\s+', ' ', regex=True) # Strips out extra spaces in the column names
    out_tbl = out_tbl.apply(pd.to_numeric, errors="coerce")    # Convert values from strings to numbers
    out_tbl = out_tbl.T.groupby(level=0).sum().T  # Groups columns with the same name and sums them together
    return out_tbl




## Processing functions

In [ ]:

def hh_size_processing(in_tbl,out_tbl,avg_over_7_per_hh):

    # Import HH Size table
    df = in_tbl


    # Transpose and clean table
    df_transposed = pd.DataFrame()
    df_transposed = transpose_tbl(df).reset_index()

    # Create fields by number of people in the household
    df_transposed['1_Per_HHs']  = df_transposed['1-person household']
    df_transposed['2_Per_HHs']  = df_transposed['2-person household']
    df_transposed['3_Per_HHs']  = df_transposed['3-person household']
    df_transposed['4_Per_HHs']  = df_transposed['4-person household']
    df_transposed['5_Per_HHs']  = df_transposed['5-person household']
    df_transposed['6_Per_HHs']  = df_transposed['6-person household']
    df_transposed['7+_Per_HHs'] = df_transposed['7-or-more person household']
    df_transposed['Total_HHs']  = (df_transposed['1-person household'] + df_transposed['2-person household'] + df_transposed['3-person household'] 
                                     +  df_transposed['4-person household'] + df_transposed['5-person household'] + df_transposed['6-person household'] 
                                     +  df_transposed['7-or-more person household'])


    # Drop all fields except for the new fields and the "Label (Grouping)" field
    df_transposed = df_transposed[['index', '1_Per_HHs', '2_Per_HHs', '3_Per_HHs', '4_Per_HHs', '5_Per_HHs', '6_Per_HHs', '7+_Per_HHs', 'Total_HHs']]
    # Change the "Label (Grouping)" field to "Blockgroup" field
    df_transposed = df_transposed.rename(columns={'index': 'Blockgroup'})


    # Calculate average household size by blockgroup
    avg_7_per_hh = avg_over_7_per_hh  # Average household size for households with 7 or more people
    hh_count_cols = ['1_Per_HHs', '2_Per_HHs', '3_Per_HHs', '4_Per_HHs','5_Per_HHs', '6_Per_HHs', '7+_Per_HHs', 'Total_HHs']

    df_transposed[hh_count_cols] = df_transposed[hh_count_cols].apply(
        lambda col: pd.to_numeric(
            col.astype(str).str.replace(',', '', regex=False),
            errors='coerce'
        )
    )

    df_transposed['Avg_HH_Size'] = (
        df_transposed['1_Per_HHs'] * 1
        + df_transposed['2_Per_HHs'] * 2
        + df_transposed['3_Per_HHs'] * 3
        + df_transposed['4_Per_HHs'] * 4
        + df_transposed['5_Per_HHs'] * 5
        + df_transposed['6_Per_HHs'] * 6
        + df_transposed['7+_Per_HHs'] * avg_7_per_hh
    ) / df_transposed['Total_HHs'].replace(0, np.nan)

    # Drop NaN rows
    df_transposed = df_transposed.dropna(subset=['Avg_HH_Size'])

    # Calculate 4+ person households by blockgroup
    df_transposed['4+_Per_HHs'] = df_transposed['4_Per_HHs'] + df_transposed['5_Per_HHs'] + df_transposed['6_Per_HHs'] + df_transposed['7+_Per_HHs']

    # Calculate distribution of household sizes by blockgroup
    df_transposed['1_Per_HHs_Distribution'] = df_transposed['1_Per_HHs'] / df_transposed['Total_HHs']
    df_transposed['2_Per_HHs_Distribution'] = df_transposed['2_Per_HHs'] / df_transposed['Total_HHs']
    df_transposed['3_Per_HHs_Distribution'] = df_transposed['3_Per_HHs'] / df_transposed['Total_HHs']
    df_transposed['4+_Per_HHs_Distribution'] = df_transposed['4+_Per_HHs'] / df_transposed['Total_HHs']



    # Create output table from regression of household size distribution by blockgroup
    # HHSIZE values from 1.0 to 4.2
    out_table = pd.DataFrame({
        "HHSIZE": np.round(np.arange(1.0, 4.21, 0.1), 1)
    })

    cols = {
        "HH1": "1_Per_HHs_Distribution",
        "HH2": "2_Per_HHs_Distribution",
        "HH3": "3_Per_HHs_Distribution",
        "HH4": "4+_Per_HHs_Distribution"
    }

    # Generate predictions
    for new_col, source_col in cols.items():

        x = df_transposed["Avg_HH_Size"]
        y = df_transposed[source_col]

        slope, intercept = np.polyfit(x, y, 1)

        out_table[new_col] = (
            slope * out_table["HHSIZE"] + intercept
        )

    # Prevent negative percentages
    hh_cols = ["HH1", "HH2", "HH3", "HH4"]

    out_table[hh_cols] = out_table[hh_cols].clip(lower=0)

    # Normalize so every row sums to exactly 1
    out_table[hh_cols] = out_table[hh_cols].div(
        out_table[hh_cols].sum(axis=1),
        axis=0
    )


    out_table.to_csv(out_tbl, index=False)








In [84]:

def hh_inc_processing(in_tbl,out_tbl,break_1,break_2,break_3,county_mean_inc):

    # Import HH Size table
    df = in_tbl


    # Transpose and clean table
    df_transposed = pd.DataFrame()
    df_transposed = transpose_tbl(df).reset_index()


    # Define county mean income
    county_mean_income = county_mean_inc

    # Define break points for income groups
    inc_group1 = break_1
    inc_group2 = break_2
    inc_group3 = break_3

    # Map original ACS income columns to their income bounds
    income_bins = {
        'Less than $10,000': (0, 10000),
        '$10,000 to $14,999': (10000, 15000),
        '$15,000 to $19,999': (15000, 20000),
        '$20,000 to $24,999': (20000, 25000),
        '$25,000 to $29,999': (25000, 30000),
        '$30,000 to $34,999': (30000, 35000),
        '$35,000 to $39,999': (35000, 40000),
        '$40,000 to $44,999': (40000, 45000),
        '$45,000 to $49,999': (45000, 50000),
        '$50,000 to $59,999': (50000, 60000),
        '$60,000 to $74,999': (60000, 75000),
        '$75,000 to $99,999': (75000, 100000),
        '$100,000 to $124,999': (100000, 125000),
        '$125,000 to $149,999': (125000, 150000),
        '$150,000 to $199,999': (150000, 200000),
        '$200,000 or more': (200000, float('inf'))
    }


    # Calculate income index for each blockgroup
    income_values = {
        col: (low + high) / 2 if high != float('inf') else 250000
        for col, (low, high) in income_bins.items()
    }
    # Calculate average household income for each row
    income_cols = list(income_bins.keys())
    df_transposed['Avg_HH_Income'] = (
        df_transposed[income_cols]
        .mul(pd.Series(income_values), axis=1)
        .sum(axis=1)
        / df_transposed[income_cols].sum(axis=1)
    )
    # Income index: county average household income = 1.0
    df_transposed['INC_INDEX'] = (
        df_transposed['Avg_HH_Income'] / county_mean_income
    )



    # Combine incomes into 4 groups based on the defined break points
    def combine_income_bins(df, lower, upper):
        cols = [
            col for col, (bin_low, bin_high) in income_bins.items()
            if bin_low >= lower and bin_high <= upper
        ]
        return df[cols].sum(axis=1)
    # Create income groups
    breaks = [0, inc_group1, inc_group2, inc_group3, float('inf')]
    for lower, upper in zip(breaks[:-1], breaks[1:]):

        if lower == 0:
            name = f'<${upper/1000:g}k'
        elif upper == float('inf'):
            name = f'${lower/1000:g}k+'
        else:
            name = f'${lower/1000:g}k-${upper/1000:g}k'

        df_transposed[name] = combine_income_bins(
            df_transposed, lower, upper
        )

    df_transposed['Total_HHs']  = (df_transposed['Less than $10,000'] + df_transposed['$10,000 to $14,999'] + df_transposed['$15,000 to $19,999']
                                     +  df_transposed['$20,000 to $24,999'] + df_transposed['$25,000 to $29,999'] + df_transposed['$30,000 to $34,999']
                                     +  df_transposed['$35,000 to $39,999'] + df_transposed['$40,000 to $44,999'] + df_transposed['$45,000 to $49,999']
                                     +  df_transposed['$50,000 to $59,999'] + df_transposed['$60,000 to $74,999'] + df_transposed['$75,000 to $99,999']
                                     +  df_transposed['$100,000 to $124,999'] + df_transposed['$125,000 to $149,999'] + df_transposed['$150,000 to $199,999'] 
                                     + df_transposed['$200,000 or more'])
    

    # Drop all fields except for the new fields and the "Label (Grouping)" field
    df_transposed = df_transposed[['index', f'<${inc_group1/1000:g}k', f'${inc_group1/1000:g}k-${inc_group2/1000:g}k', 
                                            f'${inc_group2/1000:g}k-${inc_group3/1000:g}k', f'${inc_group3/1000:g}k+','Total_HHs','Avg_HH_Income','INC_INDEX']]    
    # Change the "Label (Grouping)" field to "Blockgroup" field
    df_transposed = df_transposed.rename(columns={'index': 'Blockgroup'})
    
    
    # Calculate distribution of household incomes by blockgroup
    df_transposed['Inc1_Distribution'] = df_transposed[f'<${inc_group1/1000:g}k'] / df_transposed['Total_HHs']
    df_transposed['Inc2_Distribution'] = df_transposed[f'${inc_group1/1000:g}k-${inc_group2/1000:g}k'] / df_transposed['Total_HHs']
    df_transposed['Inc3_Distribution'] = df_transposed[f'${inc_group2/1000:g}k-${inc_group3/1000:g}k'] / df_transposed['Total_HHs']
    df_transposed['Inc4_Distribution'] = df_transposed[f'${inc_group3/1000:g}k+'] / df_transposed['Total_HHs']

    # Drop NaN rows
    df_transposed = df_transposed.dropna(subset=['INC_INDEX'])


    # Create output table from regression of household size distribution by blockgroup
    # HHINC values from 0.2 to 2
    out_table = pd.DataFrame({
        "HHINC": np.round(np.arange(0.2, 2.01, 0.1), 1)
    })

    cols = {
        "HHINC1": "Inc1_Distribution",
        "HHINC2": "Inc2_Distribution",
        "HHINC3": "Inc3_Distribution",
        "HHINC4": "Inc4_Distribution"
    }

    # Generate predictions
    for new_col, source_col in cols.items():

        x = df_transposed["INC_INDEX"]
        y = df_transposed[source_col]

        slope, intercept = np.polyfit(x, y, 1)

        out_table[new_col] = (
            slope * out_table["HHINC"] + intercept
        )

    # Prevent negative percentages
    inc_cols = ["HHINC1", "HHINC2", "HHINC3", "HHINC4"]

    out_table[inc_cols] = out_table[inc_cols].clip(lower=0)

    # Normalize so every row sums to exactly 1
    out_table[inc_cols] = out_table[inc_cols].div(
        out_table[inc_cols].sum(axis=1),
        axis=0
    )


    out_table.to_csv(out_tbl, index=False)





    


In [ ]:

def hh_worker_processing(in_tbl,out_tbl):

    # Import HH Worker table
    df = in_tbl


    # Transpose and clean table
    df_transposed = pd.DataFrame()
    df_transposed = transpose_tbl(df).reset_index()


#
#    
#
#    df_transposed['Total_HHs']  = (df_transposed['Less than $10,000'] + df_transposed['$10,000 to $14,999'] + df_transposed['$15,000 to $19,999']
#                                     +  df_transposed['$20,000 to $24,999'] + df_transposed['$25,000 to $29,999'] + df_transposed['$30,000 to $34,999']
#                                     +  df_transposed['$35,000 to $39,999'] + df_transposed['$40,000 to $44,999'] + df_transposed['$45,000 to $49,999']
#                                     +  df_transposed['$50,000 to $59,999'] + df_transposed['$60,000 to $74,999'] + df_transposed['$75,000 to $99,999']
#                                     +  df_transposed['$100,000 to $124,999'] + df_transposed['$125,000 to $149,999'] + df_transposed['$150,000 to $199,999'] 
#                                     + df_transposed['$200,000 or more'])
#    
#
#    # Drop all fields except for the new fields and the "Label (Grouping)" field
#    df_transposed = df_transposed[['index', f'<${inc_group1/1000:g}k', f'${inc_group1/1000:g}k-${inc_group2/1000:g}k', 
#                                            f'${inc_group2/1000:g}k-${inc_group3/1000:g}k', f'${inc_group3/1000:g}k+','Total_HHs','Avg_HH_Income','INC_INDEX']]    
#    # Change the "Label (Grouping)" field to "Blockgroup" field
#    df_transposed = df_transposed.rename(columns={'index': 'Blockgroup'})
#    
#    
#    # Calculate distribution of household incomes by blockgroup
#    df_transposed['Inc1_Distribution'] = df_transposed[f'<${inc_group1/1000:g}k'] / df_transposed['Total_HHs']
#    df_transposed['Inc2_Distribution'] = df_transposed[f'${inc_group1/1000:g}k-${inc_group2/1000:g}k'] / df_transposed['Total_HHs']
#    df_transposed['Inc3_Distribution'] = df_transposed[f'${inc_group2/1000:g}k-${inc_group3/1000:g}k'] / df_transposed['Total_HHs']
#    df_transposed['Inc4_Distribution'] = df_transposed[f'${inc_group3/1000:g}k+'] / df_transposed['Total_HHs']
#
#    # Drop NaN rows
#    df_transposed = df_transposed.dropna(subset=['INC_INDEX'])
#
#
#    # Create output table from regression of household size distribution by blockgroup
#    # HHINC values from 0.2 to 2
#    out_table = pd.DataFrame({
#        "HHINC": np.round(np.arange(0.2, 2.01, 0.1), 1)
#    })
#
#    cols = {
#        "HHINC1": "Inc1_Distribution",
#        "HHINC2": "Inc2_Distribution",
#        "HHINC3": "Inc3_Distribution",
#        "HHINC4": "Inc4_Distribution"
#    }
#
#    # Generate predictions
#    for new_col, source_col in cols.items():
#
#        x = df_transposed["INC_INDEX"]
#        y = df_transposed[source_col]
#
#        slope, intercept = np.polyfit(x, y, 1)
#
#        out_table[new_col] = (
#            slope * out_table["HHINC"] + intercept
#        )
#
#    # Prevent negative percentages
#    inc_cols = ["HHINC1", "HHINC2", "HHINC3", "HHINC4"]
#
#    out_table[inc_cols] = out_table[inc_cols].clip(lower=0)
#
#    # Normalize so every row sums to exactly 1
#    out_table[inc_cols] = out_table[inc_cols].div(
#        out_table[inc_cols].sum(axis=1),
#        axis=0
#    )


    out_table.to_csv(out_tbl, index=False)

In [ ]:
# Run functions

# HH Size
hhsize_in  = pd.read_csv(os.getcwd() + '/ACS/ACS_2024_B11016_HHSize.csv')
hhsize_out = os.getcwd() + '/Outputs/HHSizeModel.csv'
hh_size_processing(hhsize_in, hhsize_out, 7.2)

# HH Income
hhinc_in  = pd.read_csv(os.getcwd() + '/ACS/ACS_2024_B19001_HHInc.csv')
hhinc_out = os.getcwd() + '/Outputs/HHIncomeModel.csv'
hh_inc_processing(hhinc_in, hhinc_out, 25000, 50000, 100000, 120511)





#Below is old reference code

## Process Link Counts

In [ ]:
# Function for checking for 4 consecutive zeros and flagging those rows
def has_4_consecutive_zeros(df, row_index, n=4):
    #if row_index >= len(df):  # Ensure the row index is valid
    #    return 0  

    columns=map(str, range(24))

    row_values = df.loc[row_index, columns].astype(str).values  # Convert row values to strings
    row_str = ''.join(row_values)  # Join values into a single string
    return 1 if '0' * n in row_str else 0

# Function for checking for 8 consecutive zeros and flagging those rows
def has_8_consecutive_zeros(df, row_index, n=8):
    #if row_index >= len(df):  # Ensure the row index is valid
    #    return 0  

    columns=map(str, range(24))

    row_values = df.loc[row_index, columns].astype(str).values  # Convert row values to strings
    row_str = ''.join(row_values)  # Join values into a single string
    return 1 if '0' * n in row_str else 0


# Function for checking if row 'a' is >30% different from row 'a+1' by hour
def is_hrly_row_30_percent_different(df, row_index, threshold=0.3):

    columns = [str(i) for i in range(6,17)]

    row1 = df.loc[row_index, columns].astype(float)
    row2 = df.loc[row_index + 1, columns].astype(float)

    # Take average of row1 and row2 for denominator
    denominator = (row1 + row2)/2

    # Avoid division by zero by replacing 0 in row2 with a small value
    denominator = denominator.replace(0, 1e-9)

    # Calculate relative difference
    diff_percentage = abs(row1 - row2) / abs(denominator)

    return 1 if (diff_percentage > threshold).any() else 0  # Return 1 if any column exceeds threshold


# Function for checking if row 'a' is >30% different from row 'a+1' by daily volume
def is_dly_row_30_percent_different(df, row_index, threshold=0.3):

    row1 = df.loc[row_index, '0':'23'].sum()
    row2 = df.loc[row_index+1, '0':'23'].sum()

    # Convert to float for division
    row1 = float(row1)
    row2 = float(row2)

    # Take average of row1 and row2 for denominator
    denominator = (row1 + row2)/2
    denominator = float(denominator)

    # Avoid division by zero by replacing 0 in row2 with a small value
    if denominator == 0:
        denominator = 1e-9
    else:
        denominator = denominator

    # Calculate relative difference
    diff_percentage = abs(row1 - row2) / abs(denominator)

    return 1 if diff_percentage > threshold else 0




In [ ]:
# Function to pull link counts

# Everything below this in function
def link_cleanup(input_path,output_path):
    
    # Get a list of .xlsx files in the folder
    xlsx_files = glob.glob(os.path.join(input_path, '*.xlsx'))
    
    # Initialize dfs list to hold processed dataframes
    auto_processed_dfs = []
    bus_processed_dfs  = []
    sut_processed_dfs  = []
    mut_processed_dfs  = []
    
    # Loop through the xlsx files
    for file_path in xlsx_files:
        file_name = os.path.basename(file_path)
        print(f"Processing file: {file_name}")
        
        # Read in tabs from workbook
        # Check for "Data" tab
        try:
            df = pd.read_excel(input_path + file_name, sheet_name='Data')
        except:
            print("Data sheet not found")
            sys.exit("Terminating the script because of an invalid tab name.")

        # Delete rows 0-74
        df = df.drop(index=range(0, 75))

        # Reset index
        df.reset_index(drop=True, inplace=True)

        # Check row 0 contains Date (first day)
        cell_value = df.iloc[0, 0]
        # Check if the value matches the date format
        try:
            parsed_date_1 = datetime.strptime(cell_value, '%A, %B %d, %Y')
            print("Row 0 cell contains a valid date:", parsed_date_1)
        except ValueError:
            print("THE ZERO cell does not contain a valid date in the specified format.")
            sys.exit("Terminating the script because of an invalid date format in the ZERO cell.")
        
        # Check row 38 contains Date (second day)
        cell_value = df.iloc[38, 0]
        # Check if the value matches the date format
        try:
            parsed_date_2 = datetime.strptime(cell_value, '%A, %B %d, %Y')
            print("Row 38 cell contains a valid date:", parsed_date_2)
        except ValueError:
            print("THIRTY-EIGHT cell does not contain a valid date in the specified format.")
            sys.exit("Terminating the script because of an invalid date format in the THIRTY-EIGHT cell.")

        # Check row 76 for "Virtual Day" text
        try:
            cell_value = df.iloc[76, 0]  # Row 76, column 0
            if "Virtual Day" in cell_value:
                print("Row 76 contains 'Virtual Day'.")
            else:
                print("Row 76 does not contain 'Virtual Day'.")
                sys.exit("Terminating the script because 'Virtual Day' is missing in row 76.")
        except IndexError:
            print("Row 76 does not exist in the DataFrame.")
            sys.exit("Terminating the script due to a missing row.")

        # Drop columns 1-3 (AB, BA, & Total)
        df = df.drop(df.columns[[1,2,3]], axis=1)

        # Rename columnns
        df.columns = ['Time', 'Class1', 'Class2', 'Class3', 'Class4', 'Class5', 'Class6', 'Class7', 'Class8',
                      'Class9', 'Class10', 'Class11', 'Class12', 'Class13']

        # Aggregate by class
        df['Auto'] = df['Class1'] + df['Class2'] + df['Class3']
        df['Bus'] = df['Class4']
        df['SUT'] = df['Class5'] + df['Class6'] + df['Class7']
        df['MUT'] = df['Class8'] + df['Class9'] + df['Class10'] + df['Class11'] + df['Class12'] + df['Class13']
        
        # Split each day out of the table to make separate dataframes
        df_1 = df.iloc[5:29] # First day in the .xlsx
        df_1.reset_index(drop=True, inplace=True)
        df_1 = df_1.copy()
        df_2 = df.iloc[43:67] # Second day in the .xlsx
        df_2.reset_index(drop=True, inplace=True)
        df_2 = df_2.copy()

        # Check for 24 hours of data (first row is 0000 and last row is 2300)
        try:
            cell_value_0 = df_1.iloc[0, 0]  # Row 0, column 0
            cell_value_23 = df_1.iloc[23, 0]  # Row 23, column 0
            if ("0000" in cell_value_0) and ("2300" in cell_value_23):
                print("24 Hours of Data Present")
            else:
                print("MISSING HOURS")
                sys.exit("Terminating the script because hours are missing from the count set")
        except IndexError:
            print("Error in checking for 24 hours of data")
            sys.exit("Terminating the script due error in checking for 24 hours of data.")

        # AT THIS POINT, DATA IS IN DATABASE READABLE FORMAT
        print(df_1)

        # Loop thru each day's dataframe for processing (df_1, df_2)
        list_of_dfs = [df_1, df_2]
        list_of_dates = [parsed_date_1,parsed_date_2]
        i = 0
        for df_x in list_of_dfs:
            # Add StationName field
            df_x['StationName'] = file_name

            # Add Hour field
            df_x['Hour'] = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23]

            # Drop Class1-13 fields
            df_x = df_x.drop(df_x.columns[[1,2,3,4,5,6,7,8,9,10,11,12,13]], axis=1)

            # Split classifications out into separate dataframes for pivoting
            auto_df = df_x[['StationName','Hour','Auto']]
            bus_df  = df_x[['StationName','Hour','Bus']]
            sut_df  = df_x[['StationName','Hour','SUT']]
            mut_df  = df_x[['StationName','Hour','MUT']]

            # Pivot to get 1 row by Count_Name, 1 column by hour
            auto_df = auto_df.pivot(index='StationName', columns='Hour', values='Auto').reset_index()
            bus_df  =  bus_df.pivot(index='StationName', columns='Hour', values='Bus').reset_index()
            sut_df  =  sut_df.pivot(index='StationName', columns='Hour', values='SUT').reset_index()
            mut_df  =  mut_df.pivot(index='StationName', columns='Hour', values='MUT').reset_index()

            # Drop .xlsx from 'StationName' field
            auto_df['StationName'] = auto_df['StationName'].str.replace(r'\.xlsx$', '', regex=True)
            bus_df['StationName']  =  bus_df['StationName'].str.replace(r'\.xlsx$', '', regex=True)
            sut_df['StationName']  =  sut_df['StationName'].str.replace(r'\.xlsx$', '', regex=True)
            mut_df['StationName']  =  mut_df['StationName'].str.replace(r'\.xlsx$', '', regex=True)

            # Loop thru new vehicle dataframes
            vehicle_dfs = [auto_df, bus_df, sut_df, mut_df]
            for df_v in vehicle_dfs:
                # Add date field
                df_v['Date'] = list_of_dates[i]

                # Determine Day of Week, Month, and Year from date
                df_v['DoW']   = df_v['Date'].apply(lambda x: x.weekday())
                df_v['Month'] = df_v['Date'].apply(lambda x: x.month)
                df_v['Year']  = df_v['Date'].apply(lambda x: x.year)

                # Aggregate into periods 
                # (WILL NEED TO BE CHANGED) WILL HAVE AM, PM, OP, PMPK HOUR
                df_v['AM_Period'] = df_v.loc[:,7:8].sum(axis=1)
                #df_v['MD_Period'] = df_v.loc[:,9:15].sum(axis=1)
                df_v['PM_Period'] = df_v.loc[:,14:16].sum(axis=1)
                #df_v['NT_Period'] = df_v.loc[:,18:23].sum(axis=1) + df_v.loc[:,0:6].sum(axis=1)
                df_v['OP_Period'] = df_v.loc[:,0:6].sum(axis=1) + df_v.loc[:,9:13].sum(axis=1) + df_v.loc[:,17:23].sum(axis=1) 
                df_v['AM_PK_HR']  = df_v.loc[:,7:7].sum(axis=1)
                df_v['PM_PK_HR']  = df_v.loc[:,16:16].sum(axis=1)
                df_v['Daily'] = df_v.loc[:,0:23].sum(axis=1)

                #print(df_v)
                # Add to df lists by vehicle type
                if df_v is auto_df:
                    auto_processed_dfs.append(df_v)
                elif df_v is bus_df:
                    bus_processed_dfs.append(df_v)
                elif df_v is sut_df:
                    sut_processed_dfs.append(df_v)
                elif df_v is mut_df:
                    mut_processed_dfs.append(df_v)

            # Add 1 to i for date tagging
            i = i + 1
    
    # Combine list of dataframes (dfs) into 1 dataframe for export to csv
    auto_final_df = pd.concat(auto_processed_dfs, ignore_index = True)
    bus_final_df  = pd.concat(bus_processed_dfs,  ignore_index = True)
    sut_final_df  = pd.concat(sut_processed_dfs,  ignore_index = True)
    mut_final_df  = pd.concat(mut_processed_dfs,  ignore_index = True)

    # Convert all column names to strings for identification in factoring
    auto_final_df.columns = auto_final_df.columns.map(str)

    # Flag stations with possible erroneous counts
    for a in range(0,len(auto_final_df)):
        if a % 2 == 0: # Even, first row by station
            if auto_final_df.loc[a, '0':'23'].sum() > 1000:
                # Apply has 4 zeros function to each row, checking only columns '0' to '23'
                auto_final_df.at[a,'Has_4_zeros'] = has_4_consecutive_zeros(auto_final_df, a)
                # Apply has 8 zeros function to each row, checking only columns '0' to '23'
                auto_final_df.at[a,'Has_8_zeros'] = has_8_consecutive_zeros(auto_final_df, a)
                # Apply 30% difference threshold function to each row, checking only columns '6' to '17'
                auto_final_df.at[a,'Hourly_30%_difference_found'] = is_hrly_row_30_percent_different(auto_final_df, a)
                # Apply 30% difference threshold function to each row, checking daily
                auto_final_df.at[a,'Daily_30%_difference_found']  = is_dly_row_30_percent_different(auto_final_df, a)
            else:
                auto_final_df.at[a,'Has_4_zeros'] = 0
                # Apply has 8 zeros function to each row, checking only columns '0' to '23'
                auto_final_df.at[a,'Has_8_zeros'] = has_8_consecutive_zeros(auto_final_df, a)
                auto_final_df.at[a,'Hourly_30%_difference_found'] = 0
                auto_final_df.at[a,'Daily_30%_difference_found'] = 0
        elif a % 2 != 0: # Odd, second row by station
            if auto_final_df.loc[a, '0':'23'].sum() > 1000:
                # Apply has 4 zeros function to each row, checking only columns '0' to '23'
                auto_final_df.at[a,'Has_4_zeros'] = has_4_consecutive_zeros(auto_final_df, a)
                # Apply has 8 zeros function to each row, checking only columns '0' to '23'
                auto_final_df.at[a,'Has_8_zeros'] = has_8_consecutive_zeros(auto_final_df, a)
                # Apply 30% difference threshold function to each row, checking only columns '6' to '17'
                auto_final_df.at[a,'Hourly_30%_difference_found'] = auto_final_df.at[a-1,'Hourly_30%_difference_found']
                # Apply 30% difference threshold function to each row, checking daily
                auto_final_df.at[a,'Daily_30%_difference_found']  = auto_final_df.at[a-1,'Daily_30%_difference_found']
            else:
                auto_final_df.at[a,'Has_4_zeros'] = 0
                # Apply has 8 zeros function to each row, checking only columns '0' to '23'
                auto_final_df.at[a,'Has_8_zeros'] = has_8_consecutive_zeros(auto_final_df, a)
                auto_final_df.at[a,'Hourly_30%_difference_found'] = auto_final_df.at[a-1,'Hourly_30%_difference_found']
                auto_final_df.at[a,'Daily_30%_difference_found'] = auto_final_df.at[a-1,'Daily_30%_difference_found']



    # Import StationName - LinkNo Walkacross (with link type for factoring)
    locations = pd.read_csv(os.getcwd() + '/Count_Locations/Count_Locations.csv')

    # Import seasonal factors
    szn_dow_factors = pd.read_csv(os.getcwd() + '/Factors/Weekday_Seasonal_Factors.csv')

    # Loop thru class dataframes to join in LinkNo and Factor
    list_of_classdfs = [auto_final_df, bus_final_df,sut_final_df,mut_final_df]
    c = 0
    #for c, df_c in enumerate(list_of_classdfs):  
    for df_c in list_of_classdfs:
        # JOINING IN LINKNO & FACTOR ID INFO
        # Join walkacross to Count tables to get LINKNO
        df_c = df_c.merge(locations, how="left", on="StationName")

        # FACTORING setup
        for i in range(0,len(df_c)):
            if df_c.at[i,'TypeNo'] <= 20 and df_c.at[i,'AreaType'] == 'Urban':
                df_c.at[i,'Szn_Dow_Group'] = 0
            elif df_c.at[i,'TypeNo'] > 20 and df_c.at[i,'AreaType'] == 'Urban':
                df_c.at[i,'Szn_Dow_Group'] = 1
            elif df_c.at[i,'TypeNo'] <= 20 and df_c.at[i,'AreaType'] == 'Rural':
                df_c.at[i,'Szn_Dow_Group'] = 2
            elif df_c.at[i,'TypeNo'] > 20 and df_c.at[i,'AreaType'] == 'Rural':
                df_c.at[i,'Szn_Dow_Group'] = 4
            else:
                df_c.at[i,'Szn_Dow_Group'] = 99
        
        
        # SEASONAL FACTORING
        for r in range(0,len(df_c)):
            # Pull row values from class df
            factor_group = df_c.at[r,'Szn_Dow_Group'].astype(int)
            month        = df_c.at[r,'Month']
            # Pull factor from factor df using group/month as indices
            factor = szn_dow_factors.iloc[factor_group, month + 2]
            # Add factor to classification dataframe
            df_c.at[r,'Szn_Dow_Factor'] = factor
            # Convert all column names to strings for identification in factoring
            df_c.columns = df_c.columns.map(str)
            # Multiply each hourly count by factor field
            for col in range(24):  # Columns '0' to '23'
                df_c.at[r, str(col)] = df_c.at[r, str(col)] * factor
            # Multiply each period count by factor field
            df_c.at[r,'AM_Period'] = df_c.at[r,'AM_Period'] * factor
            #df_c.at[r,'MD_Period'] = df_c.at[r,'MD_Period'] * factor
            df_c.at[r,'PM_Period'] = df_c.at[r,'PM_Period'] * factor
            #df_c.at[r,'NT_Period'] = df_c.at[r,'NT_Period'] * factor
            df_c.at[r,'OP_Period'] = df_c.at[r,'OP_Period'] * factor
            df_c.at[r,'AM_PK_HR']  = df_c.at[r,'AM_PK_HR']  * factor
            df_c.at[r,'PM_PK_HR']  = df_c.at[r,'PM_PK_HR']  * factor
            df_c.at[r,'Daily']     = df_c.at[r,'Daily']     * factor

        # Keep StationName, Agency, LinkNo, and Period Counts only
        #df_c = df_c[['StationName','Agency','LinkNo','AM_Period','MD_Period','PM_Period','NT_Period','AM_PK_HR','PM_PK_HR']]

        list_of_classdfs[c] = df_c
        c = c + 1
    
    # Overwrite class dfs for export
    auto_final_df = list_of_classdfs[0]
    bus_final_df  = list_of_classdfs[1]
    sut_final_df  = list_of_classdfs[2]
    mut_final_df  = list_of_classdfs[3]
    
    # Export csv's
    auto_final_df.to_csv(output_path + 'Auto_Link_Counts.csv', index=False)
    bus_final_df.to_csv(output_path  + 'Bus_Link_Counts.csv',  index=False) 
    sut_final_df.to_csv(output_path  + 'SUT_Link_Counts.csv',  index=False)   
    mut_final_df.to_csv(output_path  + 'MUT_Link_Counts.csv',  index=False)  
    
# Run function
in_path  = os.getcwd() + '/2022-Spreadsheets/ALL_COUNTS/'
out_path  = os.getcwd() + '/Count_Outputs/'
    
link_cleanup(in_path,out_path)